# Next-Open Return Predictor — BQuant / BQL (self-diagnosing build)

Ranks current S&P 500 constituents by predicted return from the late-session entry price to the
**next regular-session open**.

## What is different in this build

Every stage prints a diagnostic panel **before** any guard fires, so a failure tells you which
column and which stage caused it rather than surfacing as `Only 0 complete model sessions`.

Specific fixes over v4:

| Problem in v4 | Fix here |
|---|---|
| BQL returned a 552-row **calendar-day** grid per ticker (weekends forward-filled) | `fill='NA'` + the panel is intersected with the benchmark's real trading calendar |
| `px_open` silently NaN → 4 features all-NaN → empty panel, no error until Cell 7 | Open field is **probed across candidate names** before the bulk pull; coverage is asserted at the stage that produces it |
| `dropna(subset=FEATURES)` emptied the panel with no explanation | Features are **auto-pruned** by coverage; the panel can never be emptied silently |
| `g = panel.groupby(...)` created before its columns existed | Groupbys are constructed at point of use |
| Capture window was a 60-second string compare that never fired | Widened, and the actual comparison is on `datetime.time` |
| Benchmark matched by hardcoded `"SPX Index"` string | Benchmark ID resolved from what BQL actually returned |
| Lookbacks (`pct_change(5)`, `rolling(20)`) ran on calendar days | Now run on true trading sessions |

## Still true, and still your problem

- **Survivorship bias.** `bq.univ.members()` returns *today's* index. Backtesting it over 18 months
  only tests companies that survived and stayed in the index. Fix needs point-in-time membership.
- **The proxy entry price is the close, not 15:55.** Bloomberg does not store a historical 15:55
  print retrievable via BQL. Exact 15:55 entries only exist for sessions where you ran the capture
  cell inside the window; everything else is labelled `PX_LAST_PROXY`.
- **Costs.** Close-to-open with 100% daily turnover. `round_trip_cost_bp` is applied in the ranking
  and the walk-forward summary reports the breakeven.

In [1]:
# ============================ Cell 1 — Config, imports, diagnostic helpers
import warnings, sys, traceback, datetime as dtm
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

try:
    import bql
except ImportError as exc:
    raise ImportError("Run inside Bloomberg BQuant; the bql package is required.") from exc

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
NY = "America/New_York"

CFG = {
    "index": "SPX Index",
    "benchmark": "SPX Index",
    "history_calendar_days": 800,        # ~550 trading sessions
    "minimum_price": 5.0,
    "minimum_train_sessions": 160,
    "test_sessions": 350,      # 40 had no statistical power; see Stage 9 power note
    "demean_target": True,     # train on session-demeaned (cross-sectional) target
    "top_n": 25,
    "round_trip_cost_bp": 12.0,
    "maximum_abs_target": 0.25,
    "winsor_quantiles": (0.005, 0.995),

    # capture window, exchange-local; compared as real times, not strings
    "capture_start": "15:45",
    "capture_end":   "16:05",
    "snapshot_file": Path("bquant_1555_exact_snapshots.pkl"),

    # BQL behaviour
    "bql_fill": "NA",                    # do NOT forward-fill across non-trading days
    "open_field_candidates": ["px_open", "open", "px_open_1d"],
    "probe_ticker": "IBM US Equity",

    # feature gate
    "min_feature_coverage": 0.60,        # prune any feature below this
    "min_labelled_rows": 5000,
    "target_fallback_to_close": True,    # if no open field exists, target next close instead
    "verbose_bql": True,
}

DIAG = {}


def stage(title):
    print("\n" + "=" * 74)
    print(title)
    print("=" * 74)


def note(msg):
    print(f"   {msg}")


def blocked(msg, hints=()):
    print("\n" + "!" * 74)
    print(f"BLOCKED: {msg}")
    for h in hints:
        print(f"   fix -> {h}")
    print("!" * 74)
    raise RuntimeError(msg)


def coverage_report(df, cols, label):
    cols = [c for c in cols if c in df.columns]
    if not cols:
        note(f"{label}: none of those columns exist")
        return pd.Series(dtype=float)
    cov = df[cols].notna().mean().sort_values()
    note(f"{label} — non-null fraction:")
    for k, v in cov.items():
        mark = "  " if v >= CFG["min_feature_coverage"] else " <-- LOW"
        print(f"      {k:<26} {v:6.4f}{mark}")
    return cov


def frame_report(df, label):
    note(f"{label}: shape={df.shape}")
    if "date" in df.columns and len(df):
        note(f"{label}: {df['date'].min()} -> {df['date'].max()}, "
             f"{df['date'].nunique():,} distinct dates")
    if "ticker" in df.columns and len(df):
        note(f"{label}: {df['ticker'].nunique():,} tickers, "
             f"{len(df) / max(df['ticker'].nunique(), 1):.1f} rows/ticker")


print("Config loaded. Stage diagnostics are ON.")
CFG

Config loaded. Stage diagnostics are ON.


{'index': 'SPX Index',
 'benchmark': 'SPX Index',
 'history_calendar_days': 800,
 'minimum_price': 5.0,
 'minimum_train_sessions': 160,
 'test_sessions': 350,
 'demean_target': True,
 'top_n': 25,
 'round_trip_cost_bp': 12.0,
 'maximum_abs_target': 0.25,
 'winsor_quantiles': (0.005, 0.995),
 'capture_start': '15:45',
 'capture_end': '16:05',
 'snapshot_file': PosixPath('bquant_1555_exact_snapshots.pkl'),
 'bql_fill': 'NA',
 'open_field_candidates': ['px_open', 'open', 'px_open_1d'],
 'probe_ticker': 'IBM US Equity',
 'min_feature_coverage': 0.6,
 'min_labelled_rows': 5000,
 'target_fallback_to_close': True,
 'verbose_bql': True}

In [2]:
# ============================ Cell 2 — BQL service, robust extractor, universe
stage("STAGE 2 — BQL service and universe")

bq = bql.Service()
note("bql.Service() OK")


def bql_items(response, labels, verbose=None):
    # Convert a BQL response into a tidy DataFrame. Reports the columns BQL
    # actually returned so a field-name change is visible instead of silent.
    verbose = CFG["verbose_bql"] if verbose is None else verbose
    items = list(response)
    if len(items) != len(labels):
        blocked(f"BQL returned {len(items)} items for {len(labels)} labels: {labels}",
                ["check the dict passed to bql.Request matches `labels`"])

    ignore = {"currency", "revision_date", "as_of_date", "period_end_date",
              "per_security_currency", "orig_ids"}
    frames = []
    for label, item in zip(labels, items):
        raw = item.df().reset_index()
        raw.columns = [str(c).strip().lower() for c in raw.columns]
        raw = raw.rename(columns={"id": "ticker"})
        if verbose:
            note(f"[{label}] BQL columns -> {raw.columns.tolist()}")
        if "ticker" not in raw.columns:
            blocked(f"BQL item '{label}' has no ID column. Got {raw.columns.tolist()}")

        dims = [c for c in ("ticker", "date") if c in raw.columns]
        if label in raw.columns:
            val = label
        else:
            cands = [c for c in raw.columns if c not in dims and c not in ignore]
            if not cands:
                blocked(f"No value column for BQL item '{label}'. Got {raw.columns.tolist()}")
            val = cands[0]
            note(f"[{label}] value column inferred as '{val}' (candidates {cands})")

        f = raw[dims + [val]].rename(columns={val: label}).drop_duplicates(dims, keep="last")
        frames.append(f)

    out = frames[0]
    for f in frames[1:]:
        keys = [c for c in ("ticker", "date") if c in out.columns and c in f.columns]
        if not keys:
            blocked("BQL items share no join key")
        out = out.merge(f, on=keys, how="outer")
    return out


members = bq.univ.members(CFG["index"])
meta_captured_at_ny = pd.Timestamp.now(tz=NY)

meta_response = bq.execute(bql.Request(members, {
    "name": bq.data.name(),
    "sector": bq.data.gics_sector_name(),
    "live_px_last": bq.data.px_last(),
}))
meta = bql_items(meta_response, ["name", "sector", "live_px_last"])
meta = meta[["ticker", "name", "sector", "live_px_last"]].drop_duplicates("ticker")
meta["live_px_last"] = pd.to_numeric(meta["live_px_last"], errors="coerce")

frame_report(meta, "meta")
if len(meta) < 400:
    blocked(f"Only {len(meta)} constituents returned; expected ~500",
            ["check index entitlement", "confirm CFG['index'] resolves in BQL"])

bench_live_resp = bq.execute(bql.Request(CFG["benchmark"], {"live_px_last": bq.data.px_last()}))
benchmark_live = bql_items(bench_live_resp, ["live_px_last"])[["ticker", "live_px_last"]]
benchmark_live["live_px_last"] = pd.to_numeric(benchmark_live["live_px_last"], errors="coerce")

# Resolve the benchmark ID from what BQL actually returned, not the config string.
BENCH_ID = benchmark_live["ticker"].iloc[0]
note(f"benchmark ID resolved as {BENCH_ID!r} (config asked for {CFG['benchmark']!r})")
if BENCH_ID != CFG["benchmark"]:
    note("NOTE: these differ. Using the BQL-returned ID everywhere downstream.")

live_snapshot_meta = pd.concat(
    [meta[["ticker", "live_px_last"]], benchmark_live], ignore_index=True
).dropna(subset=["live_px_last"]).drop_duplicates("ticker", keep="last")

DIAG["n_constituents"] = len(meta)
note(f"universe: {len(meta):,} stocks; live snapshot rows: {len(live_snapshot_meta):,}")
display(meta.head())


STAGE 2 — BQL service and universe
   bql.Service() OK
   [name] BQL columns -> ['ticker', 'name']
   [sector] BQL columns -> ['ticker', 'sector']
   [live_px_last] BQL columns -> ['ticker', 'date', 'currency', 'live_px_last']
   meta: shape=(503, 4)
   meta: 503 tickers, 1.0 rows/ticker
   [live_px_last] BQL columns -> ['ticker', 'date', 'currency', 'live_px_last']
   benchmark ID resolved as 'SPX Index' (config asked for 'SPX Index')
   universe: 503 stocks; live snapshot rows: 504


,ticker,name,sector,live_px_last
0,A UN Equity,Agilent Technologies Inc,Health Care,156.300003
1,AAPL UW Equity,Apple Inc,Information Technology,311.299988
2,ABBV UN Equity,AbbVie Inc,Health Care,261.829987
3,ABNB UW Equity,Airbnb Inc,Consumer Discretionary,185.000000
4,ABT UN Equity,Abbott Laboratories,Health Care,114.139999


In [3]:
# ============================ Cell 3 — Probe which open-price field actually works
stage("STAGE 3 — Open-price field probe")
# v4 assumed bq.data.px_open() returns data. When it silently returns NaN, four
# features go all-NaN and the panel empties with no error until the model cell.
# Probe on one ticker, cheaply, before pulling 500 names.

now_ny = pd.Timestamp.now(tz=NY)
end_date = now_ny.date().isoformat()
start_date = (now_ny.normalize() - pd.Timedelta(days=CFG["history_calendar_days"])).date().isoformat()
probe_start = (now_ny.normalize() - pd.Timedelta(days=90)).date().isoformat()

OPEN_FIELD = None
probe_range = bq.func.range(probe_start, end_date)

for cand in CFG["open_field_candidates"]:
    fn = getattr(bq.data, cand, None)
    if fn is None:
        note(f"{cand:<12} : not an attribute of bq.data")
        continue
    try:
        resp = bq.execute(bql.Request(CFG["probe_ticker"],
                                      {cand: fn(dates=probe_range, fill=CFG["bql_fill"])}))
        d = bql_items(resp, [cand], verbose=False)
        vals = pd.to_numeric(d[cand], errors="coerce")
        cov = float(vals.notna().mean()) if len(vals) else 0.0
        note(f"{cand:<12} : {len(d):>4} rows, coverage {cov:.3f}, sample {vals.dropna().head(2).tolist()}")
        if cov > 0.5:
            OPEN_FIELD = cand
            break
    except Exception as e:
        note(f"{cand:<12} : {type(e).__name__}: {e}")

DIAG["open_field"] = OPEN_FIELD or "(none)"

if OPEN_FIELD:
    note(f"USING OPEN FIELD -> {OPEN_FIELD!r}")
    TARGET_MODE = "NEXT_OPEN"
elif CFG["target_fallback_to_close"]:
    print("\n" + "*" * 74)
    print("WARNING: no working open-price field. Falling back to NEXT CLOSE as the target.")
    print("This changes the strategy from close->open to close->close. Results are NOT")
    print("comparable to an overnight-gap strategy. Set target_fallback_to_close=False to")
    print("hard-fail instead.")
    print("*" * 74)
    TARGET_MODE = "NEXT_CLOSE"
else:
    blocked("No usable open-price field in BQL",
            ["run FLDS <GO> on PX_OPEN and confirm entitlement",
             "extend CFG['open_field_candidates']",
             "set CFG['target_fallback_to_close']=True to proceed on close-to-close"])

note(f"TARGET_MODE = {TARGET_MODE}")


STAGE 3 — Open-price field probe
   px_open      :   91 rows, coverage 0.681, sample [262.05, 254.55]
   USING OPEN FIELD -> 'px_open'
   TARGET_MODE = NEXT_OPEN


In [4]:
# ============================ Cell 4 — History pull + trading-calendar repair
stage("STAGE 4 — Historical prices")

date_range = bq.func.range(start_date, end_date)
history_items = {"px_last": bq.data.px_last(dates=date_range, fill=CFG["bql_fill"])}
if OPEN_FIELD:
    history_items["px_open"] = getattr(bq.data, OPEN_FIELD)(dates=date_range, fill=CFG["bql_fill"])
labels = list(history_items.keys())

note(f"requesting {labels} from {start_date} to {end_date}")
equity_history = bql_items(bq.execute(bql.Request(members, history_items)), labels)
benchmark_history = bql_items(bq.execute(bql.Request(CFG["benchmark"], history_items)), labels,
                              verbose=False)

history = pd.concat([equity_history, benchmark_history], ignore_index=True)
history["date"] = pd.to_datetime(history["date"], errors="coerce", utc=True) \
                    .dt.tz_convert(None).dt.normalize()
for c in labels:
    history[c] = pd.to_numeric(history[c], errors="coerce")
history = history.dropna(subset=["ticker", "date"])
history = history.drop_duplicates(["ticker", "date"], keep="last")
history = history.sort_values(["ticker", "date"]).reset_index(drop=True)

note("--- RAW, before calendar repair ---")
frame_report(history, "history(raw)")
coverage_report(history, labels, "history(raw)")
wk = history["date"].dt.dayofweek.value_counts().sort_index()
note(f"rows by weekday (0=Mon,5=Sat,6=Sun): {wk.to_dict()}")
DIAG["raw_rows_per_ticker"] = len(history) / max(history["ticker"].nunique(), 1)

# --- calendar repair -------------------------------------------------------
# v4 got 552 rows/ticker over 550 calendar days: BQL returned a calendar grid.
# Real sessions = dates on which the benchmark itself printed a price.
bench_rows = history[history["ticker"] == BENCH_ID]
if bench_rows.empty:
    idx_like = [t for t in history["ticker"].unique() if "Index" in str(t)][:10]
    blocked(f"No rows for benchmark {BENCH_ID!r}",
            [f"tickers containing 'Index': {idx_like}",
             "set CFG['benchmark'] to one of those"])

trading_days = pd.Index(sorted(bench_rows.loc[bench_rows["px_last"].notna(), "date"].unique()))
note(f"benchmark trading sessions: {len(trading_days):,}")

before = len(history)
history = history[history["date"].isin(trading_days)]
history = history[history["px_last"].notna()]
note(f"calendar repair: {before:,} -> {len(history):,} rows "
     f"(removed {before - len(history):,} non-trading / empty rows)")

note("--- REPAIRED ---")
frame_report(history, "history")
coverage_report(history, labels, "history")
wk = history["date"].dt.dayofweek.value_counts().sort_index()
note(f"rows by weekday: {wk.to_dict()}")
if set(wk.index) & {5, 6}:
    note("WARNING: weekend rows still present — benchmark calendar looks wrong")

if OPEN_FIELD:
    stk = history[history["ticker"] != BENCH_ID]
    bm = history[history["ticker"] == BENCH_ID]
    note(f"stock px_open coverage    : {stk['px_open'].notna().mean():.4f}")
    note(f"benchmark px_open coverage: {bm['px_open'].notna().mean():.4f}")
    if bm["px_open"].notna().mean() < 0.5:
        note("WARNING: benchmark open is sparse — market_* intraday features will be pruned")

DIAG["n_sessions"] = int(history["date"].nunique())
DIAG["rows_per_ticker"] = len(history) / max(history["ticker"].nunique(), 1)
note(f"sessions={DIAG['n_sessions']}, rows/ticker={DIAG['rows_per_ticker']:.1f}")

need = CFG["minimum_train_sessions"] + CFG["test_sessions"] + 25
if DIAG["n_sessions"] < need:
    blocked(f"Only {DIAG['n_sessions']} trading sessions; need {need}",
            [f"raise CFG['history_calendar_days'] (currently {CFG['history_calendar_days']})",
             "or lower minimum_train_sessions / test_sessions"])
print(f"\nHistory OK: {history['date'].min().date()} -> {history['date'].max().date()}, "
      f"{len(history):,} rows")


STAGE 4 — Historical prices
   requesting ['px_last', 'px_open'] from 2024-06-11 to 2026-08-20


KeyboardInterrupt: 

In [ ]:
# ============================ Cell 5 — Exact late-session snapshot capture
stage("STAGE 5 — 15:55 snapshot capture")
# v4 compared "%H:%M" strings with start == end == "15:55", a 60-second window that
# in practice never fired, so every entry silently fell back to the close proxy.

snapshot_columns = ["ticker", "session", "entry_price", "captured_at_ny"]
if CFG["snapshot_file"].exists():
    exact_snapshots = pd.read_pickle(CFG["snapshot_file"])
    missing = [c for c in snapshot_columns if c not in exact_snapshots.columns]
    if missing:
        note(f"snapshot file malformed (missing {missing}); starting a fresh one")
        exact_snapshots = pd.DataFrame(columns=snapshot_columns)
else:
    exact_snapshots = pd.DataFrame(columns=snapshot_columns)
    note(f"no snapshot file at {CFG['snapshot_file']}; starting fresh")

capture_time_ny = meta_captured_at_ny
t_now = capture_time_ny.time()
t_lo = dtm.time.fromisoformat(CFG["capture_start"])
t_hi = dtm.time.fromisoformat(CFG["capture_end"])
inside = t_lo <= t_now <= t_hi
note(f"BQL snapshot clock {t_now.strftime('%H:%M:%S')} NY; "
     f"window {CFG['capture_start']}-{CFG['capture_end']} -> "
     f"{'INSIDE' if inside else 'OUTSIDE'}")

if inside:
    captured = live_snapshot_meta.rename(columns={"live_px_last": "entry_price"}).copy()
    captured["session"] = capture_time_ny.tz_localize(None).normalize()
    captured["captured_at_ny"] = capture_time_ny.isoformat()
    captured = captured.dropna(subset=["entry_price"])
    exact_snapshots = pd.concat([exact_snapshots, captured], ignore_index=True)
    exact_snapshots = exact_snapshots.sort_values("captured_at_ny") \
                                     .drop_duplicates(["ticker", "session"], keep="last")
    exact_snapshots.to_pickle(CFG["snapshot_file"])
    note(f"captured {len(captured):,} exact entries at {capture_time_ny:%Y-%m-%d %H:%M:%S %Z}")
else:
    note("no exact capture this run; entries will be labelled PX_LAST_PROXY")

if len(exact_snapshots):
    exact_snapshots["session"] = pd.to_datetime(exact_snapshots["session"], errors="coerce") \
                                   .dt.tz_localize(None).dt.normalize()
    exact_snapshots["entry_price"] = pd.to_numeric(exact_snapshots["entry_price"], errors="coerce")

DIAG["exact_snapshot_rows"] = len(exact_snapshots)
note(f"stored exact stock-session observations: {len(exact_snapshots):,}")

In [ ]:
# ============================ Cell 6 — Panel, target, features
stage("STAGE 6 — Panel construction")

panel = history.rename(columns={"date": "session", "px_last": "close_proxy"}).copy()
if OPEN_FIELD:
    panel = panel.rename(columns={"px_open": "session_open"})
else:
    panel["session_open"] = np.nan

if len(exact_snapshots):
    panel = panel.merge(exact_snapshots[["ticker", "session", "entry_price"]],
                        on=["ticker", "session"], how="left")
else:
    panel["entry_price"] = np.nan
panel["entry_source"] = np.where(panel["entry_price"].notna(), "EXACT_1555", "PX_LAST_PROXY")
panel["entry_price"] = panel["entry_price"].fillna(panel["close_proxy"])
panel = panel.sort_values(["ticker", "session"]).reset_index(drop=True)

# --- next-session calendar -------------------------------------------------
market_sessions = np.array(sorted(panel.loc[panel["ticker"] == BENCH_ID, "session"].unique()))
if len(market_sessions) < 2:
    blocked("benchmark produced no usable trading calendar")
calendar = pd.DataFrame({"session": market_sessions[:-1], "next_session": market_sessions[1:]})
panel = panel.merge(calendar, on="session", how="left")

target_col = "session_open" if TARGET_MODE == "NEXT_OPEN" else "close_proxy"
nxt = panel[["ticker", "session", target_col]].rename(
    columns={"session": "next_session", target_col: "next_px"})
panel = panel.merge(nxt, on=["ticker", "next_session"], how="left")
panel["target_next_open"] = panel["next_px"] / panel["entry_price"] - 1

note(f"target = next {target_col} / entry_price - 1   [{TARGET_MODE}]")
for c in ["entry_price", "session_open", "next_px", "target_next_open"]:
    note(f"panel.{c:<18} non-null {panel[c].notna().mean():.4f}")

# --- features (groupbys built at point of use; v4 built one before its columns existed)
panel["ret_intraday"] = panel["entry_price"] / panel["session_open"] - 1
panel["ret_1d"]  = panel.groupby("ticker")["entry_price"].pct_change(1, fill_method=None)
panel["ret_5d"]  = panel.groupby("ticker")["entry_price"].pct_change(5, fill_method=None)
panel["ret_20d"] = panel.groupby("ticker")["entry_price"].pct_change(20, fill_method=None)
panel["ma20_gap"] = panel["entry_price"] / panel.groupby("ticker")["entry_price"] \
                        .transform(lambda s: s.rolling(20, min_periods=15).mean()) - 1
panel["rv_10d"] = panel.groupby("ticker")["ret_1d"] \
                       .transform(lambda s: s.rolling(10, min_periods=8).std()) * np.sqrt(252)
panel["rv_20d"] = panel.groupby("ticker")["ret_1d"] \
                       .transform(lambda s: s.rolling(20, min_periods=15).std()) * np.sqrt(252)

bench = panel[panel["ticker"] == BENCH_ID][
    ["session", "ret_intraday", "ret_1d", "ret_5d", "rv_20d"]].copy()
bench = bench.rename(columns={c: f"market_{c}" for c in bench.columns if c != "session"})
panel = panel.merge(bench, on="session", how="left")
panel["relative_intraday"] = panel["ret_intraday"] - panel["market_ret_intraday"]
panel["relative_5d"] = panel["ret_5d"] - panel["market_ret_5d"]

for c in ["ret_intraday", "ret_1d", "ret_5d", "ret_20d", "ma20_gap", "rv_20d"]:
    panel[f"rank_{c}"] = panel.groupby("session")[c].rank(pct=True) - 0.5

model_df = panel[panel["ticker"] != BENCH_ID].merge(
    meta[["ticker", "name", "sector"]], on="ticker", how="inner")
bad = model_df["target_next_open"].abs() > CFG["maximum_abs_target"]
model_df.loc[bad, "target_next_open"] = np.nan
model_df = model_df.replace([np.inf, -np.inf], np.nan)

# Train on the cross-sectionally demeaned target: a dollar-neutral book cannot
# harvest the common overnight gap, so predicting it wastes model capacity.
# Rank IC is unchanged by a per-session shift, so evaluation is unaffected.
if CFG["demean_target"]:
    model_df["target_train"] = model_df["target_next_open"] - \
        model_df.groupby("session")["target_next_open"].transform("mean")
    note("target_train = session-demeaned target_next_open")
else:
    model_df["target_train"] = model_df["target_next_open"]
    note("target_train = raw target_next_open")

note(f"extreme/corporate-action targets nulled: {int(bad.sum()):,}")
frame_report(model_df.rename(columns={"session": "date"}), "model_df")
DIAG["model_df_rows"] = len(model_df)

In [ ]:
# ============================ Cell 7 — Feature health gate (auto-prune)
stage("STAGE 7 — Feature health gate")
# The v4 failure mode: dropna(subset=ALL_FEATURES) silently emptied the panel because
# one column was 100% NaN. Here low-coverage features are pruned and reported instead.

CANDIDATE_FEATURES = [
    "ret_intraday", "ret_1d", "ret_5d", "ret_20d", "ma20_gap", "rv_10d", "rv_20d",
    "market_ret_intraday", "market_ret_1d", "market_ret_5d", "market_rv_20d",
    "relative_intraday", "relative_5d", "rank_ret_intraday", "rank_ret_1d",
    "rank_ret_5d", "rank_ret_20d", "rank_ma20_gap", "rank_rv_20d",
]

absent = [c for c in CANDIDATE_FEATURES if c not in model_df.columns]
if absent:
    note(f"not present in model_df, skipped: {absent}")
present = [c for c in CANDIDATE_FEATURES if c in model_df.columns]

cov = coverage_report(model_df, present + ["target_next_open", "target_train"], "candidate features")

FEATURES = [c for c in present if cov.get(c, 0.0) >= CFG["min_feature_coverage"]]
pruned = [c for c in present if c not in FEATURES]
if pruned:
    print()
    note(f"PRUNED {len(pruned)} low-coverage feature(s): {pruned}")
    if not OPEN_FIELD:
        note("(expected: no open field, so all intraday-derived features are empty)")
if not FEATURES:
    blocked("Every candidate feature failed the coverage gate",
            ["inspect the coverage table above",
             "lower CFG['min_feature_coverage']"])

note(f"KEEPING {len(FEATURES)} features: {FEATURES}")

# --- cumulative row loss, so an empty panel is never a mystery -------------
print()
note("cumulative row loss through dropna:")
surv = model_df.copy()
culprit = None
for c in FEATURES + ["target_next_open", "target_train"]:
    b = len(surv)
    surv = surv.dropna(subset=[c])
    if len(surv) < b:
        print(f"      {c:<26} {b:>9,} -> {len(surv):>9,}  (-{b - len(surv):,})")
    if len(surv) == 0:
        culprit = c
        break

if culprit:
    blocked(f"Panel emptied at feature '{culprit}'",
            [f"'{culprit}' coverage is {cov.get(culprit, float('nan')):.4f}",
             "raise CFG['min_feature_coverage'] to prune it automatically"])

labelled = surv.copy()
sessions = np.array(sorted(labelled["session"].unique()))
frame_report(labelled.rename(columns={"session": "date"}), "labelled")
note(f"complete sessions: {len(sessions):,}")

# --- adapt the walk-forward split to what actually exists ------------------
required = CFG["minimum_train_sessions"] + CFG["test_sessions"]
if len(sessions) < required:
    if len(sessions) < 60:
        blocked(f"Only {len(sessions)} complete sessions — too few to model",
                ["see the cumulative row-loss table above for what is dropping rows",
                 "raise CFG['history_calendar_days']"])
    TEST_SESSIONS = max(10, int(len(sessions) * 0.25))
    MIN_TRAIN = len(sessions) - TEST_SESSIONS
    note(f"ADAPTED: {len(sessions)} sessions available (< {required} configured); "
         f"using min_train={MIN_TRAIN}, test={TEST_SESSIONS}")
else:
    TEST_SESSIONS = CFG["test_sessions"]
    MIN_TRAIN = CFG["minimum_train_sessions"]
    note(f"split: min_train={MIN_TRAIN}, test={TEST_SESSIONS}")

DIAG["n_features"] = len(FEATURES)
DIAG["labelled_rows"] = len(labelled)
DIAG["complete_sessions"] = len(sessions)

In [ ]:
# ============================ Cell 8 — Scoring session audit
stage("STAGE 8 — Latest scoring session")

session_counts = model_df.groupby("session")["entry_price"].count()
min_cov = int(np.ceil(0.85 * meta["ticker"].nunique()))
eligible = session_counts.index[session_counts >= min_cov]
if len(eligible) == 0:
    note(f"best session coverage: {session_counts.max()} vs required {min_cov}")
    blocked("No session reaches 85% universe coverage")

latest_session = eligible.max()
latest = model_df[model_df["session"] == latest_session]

audit = pd.Series({
    "latest_scoring_session": latest_session,
    "target_mode": TARGET_MODE,
    "open_field": OPEN_FIELD or "(none)",
    "eligible_constituents": meta["ticker"].nunique(),
    "latest_entry_obs": int(latest["entry_price"].notna().sum()),
    "latest_exact_1555": int((latest["entry_source"] == "EXACT_1555").sum()),
    "all_exact_1555": int((model_df["entry_source"] == "EXACT_1555").sum()),
    "all_proxy": int((model_df["entry_source"] == "PX_LAST_PROXY").sum()),
    "trading_sessions": DIAG["n_sessions"],
    "rows_per_ticker": round(DIAG["rows_per_ticker"], 1),
    "features_kept": DIAG["n_features"],
    "labelled_rows": DIAG["labelled_rows"],
    "complete_sessions": DIAG["complete_sessions"],
    "duplicate_ticker_sessions": int(model_df.duplicated(["ticker", "session"]).sum()),
    "nonpositive_entry_prices": int((model_df["entry_price"].dropna() <= 0).sum()),
})
display(audit.to_frame("value"))

if audit["duplicate_ticker_sessions"]:
    blocked("Duplicate ticker-session rows")
if audit["nonpositive_entry_prices"]:
    blocked("Non-positive entry prices")
staleness = (now_ny.tz_localize(None).normalize() - latest_session).days
note(f"latest session is {staleness} calendar days old")
if staleness > 5:
    blocked(f"Latest scoring session stale by {staleness} days")

In [ ]:
# ============================ Cell 9 — Walk-forward out-of-sample evaluation
stage("STAGE 9 — Walk-forward evaluation")


def make_model():
    return HistGradientBoostingRegressor(
        loss="squared_error", learning_rate=0.045, max_iter=180,
        max_leaf_nodes=15, min_samples_leaf=80, l2_regularization=2.0,
        random_state=42,
    )


parts, skipped = [], 0
for test_day in sessions[-TEST_SESSIONS:]:
    train = labelled[labelled["session"] < test_day]
    test = labelled[labelled["session"] == test_day].copy()
    if train["session"].nunique() < MIN_TRAIN or test.empty:
        skipped += 1
        continue
    lo, hi = train["target_train"].quantile(CFG["winsor_quantiles"])
    mdl = make_model().fit(train[FEATURES], train["target_train"].clip(lo, hi))
    test["prediction"] = mdl.predict(test[FEATURES])
    parts.append(test[["session", "ticker", "target_next_open", "prediction"]])

note(f"test days attempted {TEST_SESSIONS}, skipped {skipped}, scored {len(parts)}")
if not parts:
    blocked("Walk-forward produced no predictions",
            [f"MIN_TRAIN={MIN_TRAIN} but only {len(sessions)} sessions exist",
             "lower CFG['minimum_train_sessions']"])

oos = pd.concat(parts, ignore_index=True)

rows = []
for day, d in oos.groupby("session", sort=True):
    n = max(1, len(d) // 10)
    rows.append({
        "session": day,
        "spearman_ic": d["prediction"].corr(d["target_next_open"], method="spearman"),
        "top_decile": d.nlargest(n, "prediction")["target_next_open"].mean(),
        "bottom_decile": d.nsmallest(n, "prediction")["target_next_open"].mean(),
    })
daily = pd.DataFrame(rows)
daily["long_short"] = daily["top_decile"] - daily["bottom_decile"]

gross_bp = daily["long_short"].mean() * 10_000
metrics = pd.Series({
    "OOS sessions": oos["session"].nunique(),
    "OOS observations": len(oos),
    "MAE (bp)": mean_absolute_error(oos["target_next_open"], oos["prediction"]) * 10_000,
    "Mean daily Spearman IC": daily["spearman_ic"].mean(),
    "IC t-stat": (daily["spearman_ic"].mean() / daily["spearman_ic"].std()
                  * np.sqrt(len(daily))) if daily["spearman_ic"].std() else np.nan,
    "Positive IC session rate": (daily["spearman_ic"] > 0).mean(),
    "Mean top-decile return (bp)": daily["top_decile"].mean() * 10_000,
    "Mean top-minus-bottom, gross (bp)": gross_bp,
    "Round-trip cost (bp)": CFG["round_trip_cost_bp"],
    "Top-minus-bottom, net (bp)": gross_bp - CFG["round_trip_cost_bp"],
})
display(metrics.to_frame("value"))

# --- statistical power -----------------------------------------------------
ic_sd = daily["spearman_ic"].std()
n_obs = len(daily)
print(f"\nPOWER: daily IC sd = {ic_sd:.4f} over {n_obs} sessions.")
for tgt in (0.01, 0.02, 0.03):
    need = int(np.ceil((2 * ic_sd / tgt) ** 2))
    verdict = "detectable" if n_obs >= need else f"NOT detectable (need ~{need})"
    print(f"   true IC of {tgt:.2f} at t=2 -> {verdict}")
if n_obs < 200:
    print("   WARNING: too few sessions to distinguish 'no edge' from 'small real edge'.")

print("\nVERDICT:", "survives costs" if gross_bp > CFG["round_trip_cost_bp"]
      else "DOES NOT survive costs at the configured round-trip")

(daily.set_index("session")["long_short"].fillna(0).add(1).cumprod() - 1).plot(
    figsize=(11, 4), title=f"Walk-forward top-minus-bottom, gross [{TARGET_MODE}]")
plt.axhline(0, color="black", lw=0.8)
plt.show()

In [ ]:
# ============================ Cell 10 — Latest-session ranking
stage("STAGE 10 — Ranking")

train = labelled[labelled["session"] < latest_session]
score = model_df[(model_df["session"] == latest_session)
                 & (model_df["entry_price"] >= CFG["minimum_price"])] \
        .dropna(subset=FEATURES).copy()

note(f"train rows {len(train):,} over {train['session'].nunique():,} sessions")
note(f"scoreable names at {pd.Timestamp(latest_session).date()}: {len(score):,}")
if train.empty:
    blocked("No training rows before the latest session")
if score.empty:
    blocked("No scoreable names in the latest session",
            ["check per-feature coverage for that session"])

lo, hi = train["target_train"].quantile(CFG["winsor_quantiles"])
final_model = make_model().fit(train[FEATURES], train["target_train"].clip(lo, hi))
score["predicted_return"] = final_model.predict(score[FEATURES])
score["predicted_bp"] = score["predicted_return"] * 10_000
score["net_bp"] = score["predicted_bp"] - CFG["round_trip_cost_bp"]

ranking = score.sort_values("predicted_return", ascending=False)[
    ["ticker", "name", "sector", "session", "entry_source", "entry_price",
     "predicted_bp", "net_bp"]].reset_index(drop=True)
ranking.insert(0, "rank", np.arange(1, len(ranking) + 1))

print(f"\nTop {CFG['top_n']} — entry {pd.Timestamp(latest_session).date()}, "
      f"exit next {'open' if TARGET_MODE == 'NEXT_OPEN' else 'close'}")
display(ranking.head(CFG["top_n"]).round(3))
note(f"names with positive net expectation: {(ranking['net_bp'] > 0).sum():,}")

In [ ]:
# ============================ Cell 11 — Export
stamp = pd.Timestamp(latest_session).strftime("%Y%m%d")
ranking_file = Path(f"NextOpen_Ranking_{stamp}.csv")
oos_file = Path(f"NextOpen_OOS_{stamp}.csv")
diag_file = Path(f"NextOpen_Diagnostics_{stamp}.csv")

ranking.to_csv(ranking_file, index=False)
oos.to_csv(oos_file, index=False)
pd.Series(DIAG).to_frame("value").to_csv(diag_file)

print(f"wrote {ranking_file}  ({len(ranking):,} rows)")
print(f"wrote {oos_file}      ({len(oos):,} rows)")
print(f"wrote {diag_file}")
print("\nrun diagnostics:")
print(pd.Series(DIAG).to_frame("value").to_string())

---
## Reading the diagnostics

Each stage prints before its guard fires. If the notebook stops, the block immediately above the
traceback names the stage, the column and the fix.

**`rows_per_ticker`** is the fastest health check. It should be close to `trading_sessions`
(~252/year). If it materially exceeds that, BQL is returning a calendar-day grid again and the
benchmark calendar intersection in Cell 4 didn't catch it.

**`open_field`** shows which BQL mnemonic actually returned open prices. If it reads `(none)`,
`TARGET_MODE` will be `NEXT_CLOSE` and **you are no longer testing an overnight-gap strategy** —
the intraday features are pruned and the target is close-to-close. Treat those results as a
different experiment, not as a weaker version of the same one.

**`all_exact_1555` vs `all_proxy`** is the honesty column. Exact entries only accumulate on days you
run Cell 5 inside the capture window; they build up one session at a time going forward. Until that
count is material, the backtest is a close-to-open study, not a 15:55-to-open study.

**`features_kept`** below 19 means the gate pruned something. The coverage table in Stage 7 says
which, and the cumulative row-loss table says what it cost.

## Before trusting any number

1. Costs — the verdict line in Stage 9 compares gross top-minus-bottom against the configured
   round trip. `round_trip_cost_bp = 12` is optimistic for daily rebalancing of 500 names.
2. Survivorship — current index membership applied to 18 months of history.
3. The opening auction is not a price you can reliably take; realised fills will trail the backtest.
4. Every iteration you run on this backtest spends statistical significance. Hold out the most
   recent months and leave them alone until you are finished.